# 3 · NumPy arrays

The same `Quantity` that holds a scalar also **wraps a NumPy array** (via NEP 13 / NEP 18 — a wrapper, never an `ndarray` subclass). Units ride through ufuncs, reductions and broadcasting, and are never silently dropped.

In [1]:
import numpy as np
import duq

a = np.array([1.0, 2.0, 3.0]) * duq.units.eV   # ndarray * Unit -> Quantity
print(a)
print(a.shape, a.dtype)
a[1]   # element access returns a Quantity

[1. 2. 3.] eV
(3,) float64


Quantity(np.float64(2.0), Unit('eV'))

## Ufuncs and reductions

In [2]:
d = np.array([1.0, 2.0]) * duq.units.m
t = np.array([1.0, 1.0]) * duq.units.s
print(d / t)
print(np.sqrt(d * d).unit)
print(d.sum(), d.mean())
print(d.var().unit)   # variance squares the unit

[1. 2.] m·s⁻¹
m
3.0 m 1.5 m
m²


## Joins, selections, comparisons

Same-dimension operations convert the right operand to the left unit; comparisons return plain boolean arrays.

In [3]:
print(np.concatenate([np.array([1.0]) * duq.units.m,
                      np.array([100.0]) * duq.units.cm]))
print(np.where(np.array([True, False]),
               np.array([1.0, 2.0]) * duq.units.m,
               np.array([100.0, 200.0]) * duq.units.cm))
print((np.array([1.0, 2.0]) * duq.units.m) > (np.array([50.0, 250.0]) * duq.units.cm))

[1. 1.] m
[1. 2.] m
[ True False]


## Fail-loud, never silent

Coercing to a bare array would drop the unit, so it raises with `ustrip` guidance. Dimensionless quantities *do* coerce (applying the scale).

In [4]:
np.asarray(a)

UnsupportedOperationError: refusing to convert a Quantity to a bare NumPy array, which would silently drop its unit; use duq.ustrip(unit, q) to get the magnitude in a chosen unit (or q.value for the stored magnitude)

In [5]:
print(float(duq.Quantity(50.0, "%")))   # dimensionless -> 0.5
print(duq.ustrip("J", a))                # magnitude in a chosen unit

0.5
[1.60217663e-19 3.20435327e-19 4.80652990e-19]
